In [ ]:
from pathlib import Path
import subprocess
import sys
from IPython.display import Markdown, display
import pandas as pd

# Подключаем корень проекта для импорта наших модулей из папки src
sys.path.insert(0, str(Path.cwd().parent))
from src.cleaning import run_cleaning_pipeline
from src.config import CLEAN_DATA_PATH, RAW_FILES, REVIEW_SAMPLE_PATH

# 1. Загружаем отладочный срез отзывов (200 000 строк) и таблицу заведений
df_reviews = pd.read_json(REVIEW_SAMPLE_PATH, lines=True)
df_business = pd.read_json(RAW_FILES["business"], lines=True)

# 2. Вызываем общий конвейер очистки из модуля src/cleaning.py
df_clean_notebook, summary_table = run_cleaning_pipeline(df_reviews, df_business)

display(Markdown("### Итоговая сводка «Было — Стало» по этапам очистки (Шаг 12)"))
display(summary_table)

lost_total = len(df_reviews) - len(df_clean_notebook)
lost_pct = (lost_total / len(df_reviews)) * 100
print(f"ИТОГО: Было {len(df_reviews)} строк -> Стало {len(df_clean_notebook)} строк")
print(f"Всего удалено: {lost_total} строк ({lost_pct:.3f}% от исходного объёма)")

display(df_clean_notebook.head(3))

### Итоговая сводка «Было — Стало» по этапам очистки (Шаг 12)

,Этап очистки,Строк до,Строк после,Удалено строк,"Доля потерь, %"
0,1. Соединение таблиц (JOIN),200000,200000,0,0.000
1,2. Нормализация имён столбцов,200000,200000,0,0.000
2,3. Обработка пропусков,200000,199990,10,0.005
3,4. Приведение типов и дат,199990,199990,0,0.000
4,5. Удаление дубликатов,199990,199990,0,0.000
5,6. Нормализация категорий и городов,199990,199990,0,0.000
6,7. Очистка текста отзыва,199990,199980,10,0.005
7,8. Проверка предметных ограничений,199980,199980,0,0.000
8,9. Обработка выбросов (IQR vs 3-sigma),199980,199980,0,0.000


ИТОГО: Было 200000 строк -> Стало 199980 строк
Всего удалено: 20 строк (0.010% от исходного объёма)


,review_id,user_id,business_id,stars,useful,funny,cool,text,date,business_name,...,state,postal_code,latitude,longitude,business_stars,review_count,is_open,main_category,stars_100,is_useful_outlier
0,--0HN389rkL6zLzqUrpGog,wjtj6GJ55PlAFo-bZUanSQ,iDtIMWJVIGYspg5JvJKaOw,2,2,0,0,I've been curious to try Mad Mex since it open...,2014-08-26 14:05:06,Mad Mex - Wynnewood,...,PA,19096,40.001846,-75.271637,2.5,248,False,Food,40.0,False
1,--2z5bwg3PPrWnr9wI84cA,J8x_xZIrxJd0aqG8ly0IAQ,D3DMIr-I0AKzlIKdErxHGg,5,2,0,2,This place is located where Salt used to be. T...,2014-03-03 18:20:19,Nathalie's,...,MO,63108,38.641399,-90.252411,3.0,67,False,American (Traditional),100.0,False
2,--46suwqJVAnei8ng2TeDw,yRgmyex9EXjFN5zDbxQkfw,qlC5ojQRLe26shP1Kt7hQg,3,0,0,0,I'm giving this Waffle house 3 stars because o...,2012-09-13 00:46:06,Waffle House,...,LA,70094,29.909586,-90.206871,2.5,27,True,Diners,60.0,False


In [2]:
# 1. Запускаем консольный скрипт src/pipeline.py прямо из командной строки
cli_run = subprocess.run(
    [sys.executable, "-m", "src.pipeline", "--sample"],
    cwd=str(Path.cwd().parent),
    capture_output=True,
    text=True,
    encoding="utf-8",
    check=True,
)
print("=== Вывод консольного модуля src/pipeline.py ===")
print(cli_run.stdout)

# 2. Проверка согласованности: сверяем таблицу из блокнота и файл clean.parquet от скрипта
df_clean_cli = pd.read_parquet(CLEAN_DATA_PATH)
pd.testing.assert_frame_equal(
    df_clean_notebook, df_clean_cli, check_categorical=False, check_dtype=False
)
print("✅ Проверка согласованности пройдена: результаты блокнота и модуля совпадают построчно на 100%!")

# 3. Проверка идемпотентности: запускаем скрипт второй раз подряд и проверяем, что данные не изменились
subprocess.run(
    [sys.executable, "-m", "src.pipeline", "--sample"],
    cwd=str(Path.cwd().parent),
    check=True,
)
df_clean_second = pd.read_parquet(CLEAN_DATA_PATH)
pd.testing.assert_frame_equal(df_clean_cli, df_clean_second)
print("✅ Проверка идемпотентности пройдена: повторный запуск даёт абсолютно тот же результат!")

=== Вывод консольного модуля src/pipeline.py ===
2026-11-01 21:38:12,687 | INFO | Чтение входных файлов: review_sample_200k.jsonl и yelp_academic_dataset_business.json
2026-11-01 21:38:17,197 | INFO | Этап 1 (Соединение таблиц): было 200000 строк, стало 200000 строк, не сопоставилось: 0
2026-11-01 21:38:17,204 | INFO | Этап 2 (Имена столбцов): приведено к единому виду 19 столбцов
2026-11-01 21:38:18,360 | INFO | Этап 3 (Пропуски): было 200000 строк, стало 199990 строк (удалено 10 строк без категории)
2026-11-01 21:38:18,434 | INFO | Этап 4 (Типы и даты): было 199990 строк, стало 199990 строк
2026-11-01 21:38:19,082 | INFO | Этап 5 (Дубликаты): было 199990 строк, стало 199990 строк (удалено 0 дублей)
2026-11-01 21:38:19,293 | INFO | Этап 6 (Нормализация категорий): уникальных городов=320, категорий=260
2026-11-01 21:38:23,122 | INFO | Этап 7 (Очистка текста): было 199990 строк, стало 199980 строк (удалено 10 коротких строк)
2026-11-01 21:38:23,130 | INFO | Этап 8 (Предметные правила): б